# Mt. Hood Ash Dispersal — Tephra2 Parameter Sweep (adjustable regional grid)

This notebook runs [Tephra2](https://gscommunitycodes.usf.edu/geoscicommunitycodes/public/tephra2/tephra2.php)
(an advection-diffusion tephra transport model) over a sweep of eruption source parameters (ESPs) for
**Mt. Hood**, evaluated over a regional grid around the vent (plus the 3 POIs), and aggregates the results
into a single CSV (`tephra2_aggregated.csv`) for downstream hazard analysis.

### Volcano context
Mt. Hood is a low-explosivity, dome-collapse stratovolcano. Its eruptive history is dominated by dacite dome
growth, pyroclastic flows, and lahars, with comparatively little pumiceous ash. This constrains the physically
realistic parameter ranges used below, particularly plume height — this is **not** a Plinian-eruption parameter
sweep.

### How this notebook differs from `tephra2_sweep.ipynb`
`tephra2_sweep.ipynb` evaluates only the 3 POIs (Rhododendron, Parkdale, Govt. Camp) -- no spatial grid at all,
which is what makes a large parameter sweep tractable, but gives up any regional/spatial hazard picture. This
notebook instead evaluates a *configurable* regional grid, so you can deliberately trade spatial coverage
against runtime: `grid_radius` and `grid_spacing` in the grid file cell below fully control how many grid
points there are (`(2 * grid_radius / grid_spacing) ** 2`, roughly), independent of the parameter sweep.

Tephra2's own per-run cost scales with grid points x column steps x particle steps, so **total compute scales
with `n_steps**3 * n_grid_points`** -- both knobs matter. The original all-region version of this sweep used a
200 x 200 km / 1 km grid (~40,000 points), which could not reliably finish within a single VICTOR session even
with a fairly small parameter sweep; the defaults below use a much smaller grid as a deliberately more
conservative starting point. See the grid file cell for guidance on picking `grid_radius`/`grid_spacing`.

**No matching analysis notebook yet:** `tephra2_analysis.ipynb` was rewritten around the 3-POI-only CSVs that
`tephra2_sweep.ipynb` produces and does not read this notebook's `tephra2_aggregated.csv` (single file, includes
`easting`/`northing` for every grid point). If you run this notebook, you'll need separate analysis code (or
ask for `tephra2_analysis.ipynb` to be extended) to make use of its output.

### What this notebook does
1. Sets the vent location (elevation from a downloaded DEM) and fixed ESPs.
2. Builds the Tephra2 grid file: a configurable square grid centered on the vent, plus the three POIs
   (Rhododendron, Parkdale, Govt. Camp), all at a fixed 1000 m elevation (Tephra2's advection-diffusion
   solution assumes flat topography).
3. Pulls an ERA5 reanalysis wind profile (October 2024) via `cdsapi` and formats it for Tephra2.
4. Sweeps plume height, eruption mass, and diffusion coefficient (`n_steps` steps each = `n_steps**3` Tephra2
   runs) and appends every run's grid output to `tephra2_aggregated.csv`.

This notebook builds directly on the working single-run workflow in `prelim_tephra2_working.ipynb` — the vent
location, grid construction, and wind-fetching logic are unchanged from that notebook; this notebook wraps them
in a full parameter sweep.

**Runtime note:** if a run gets interrupted (kernel death, session timeout, etc.), the run loop below resumes
from wherever `tephra2_aggregated.csv` left off rather than starting over -- just re-run the sweep cell.

In [ ]:
import os          # run tephra2 externally, remove temp files
import sys
import time
import subprocess   # run the compiled tephra2 binary

import numpy as np
import pandas as pd
import utm           # lat/lon <-> UTM conversion
import cdsapi         # ERA5 reanalysis wind data
import netCDF4        # read downloaded wind netCDF

sys.path.insert(0, "/home/jovyan/shared/Libraries/")
import victor          # VICTOR platform DEM download
import rioxarray as rxr  # read DEM geotiff for vent elevation

TEPHRA2_BIN = "/home/jovyan/tephra2/tephra2_2020"


## Vent location and elevation

Vent coordinates and elevation lookup are unchanged from `prelim_tephra2_working.ipynb`.

In [ ]:
# Mt. Hood vent coordinates
vent_latitude, vent_longitude = 45.22, -121.44

converted = utm.from_latlon(vent_latitude, vent_longitude)
vent_easting = converted[0]
vent_northing = converted[1]
utm_zone_number = converted[2]
utm_zone_letter = converted[3]

dem_name = victor.download_dem(
    vent_latitude + 1, vent_latitude - 1, vent_longitude + 1, vent_longitude - 1,
    "tiff", "SRTMGL3", filename="tephra2.tiff", api_key='73d3f5f5000048606a3c15ff635bde63')

dem = rxr.open_rasterio(dem_name)
vent_elevation = dem.sel(x=vent_longitude, y=vent_latitude, method="nearest").values[0]

print(f"Vent UTM: {vent_easting:.0f} E, {vent_northing:.0f} N (zone {utm_zone_number}{utm_zone_letter})")
print(f"Vent elevation: {vent_elevation:.0f} m")


## Fixed eruption source parameters

`PLUME_HEIGHT`, `ERUPTION_MASS`, and `DIFFUSION_COEFFICIENT` are the three swept parameters (set further down).
Everything else is held fixed at the same values used in `prelim_tephra2_working.ipynb`. A value of `0` below
means "leave blank / use Tephra2's built-in default" — see the ESP descriptions in the preliminary notebook for
definitions and units of each keyword.

In [ ]:
median_grain = 1
std_grain = 1

falltime_thresh = 0
plume_model = 0
part_steps = 0
lithic_density = 0
col_steps = 0
max_grain = 0
alpha = 0
beta = 0
min_grain = 0
pumice_density = 0
eddy_const = 0


## Grid file

A configurable square grid centered on the vent, plus the three POI locations (Rhododendron, Parkdale,
Govt. Camp). All points share a fixed elevation of 1000 m, since Tephra2's integral advection-diffusion
solution requires flat topography.

**`grid_radius`** (meters, either side of the vent) sets the domain extent -- `2 * grid_radius` is the full
square's side length. **`grid_spacing`** (meters) sets the resolution. Roughly
`n_grid_points ≈ (2 * grid_radius / grid_spacing) ** 2` (plus the 3 POIs), and that's the number that multiplies
directly into total sweep runtime alongside `n_steps**3`.

Some reference points, going from the defaults below up toward the original full-region version:

| `grid_radius` | `grid_spacing` | domain | ~points |
|---:|---:|---:|---:|
| 50,000 | 5,000 | 100 x 100 km | ~400 |
| 50,000 | 2,000 | 100 x 100 km | ~2,500 |
| 100,000 | 5,000 | 200 x 200 km | ~1,600 |
| 100,000 | 1,000 | 200 x 200 km | ~40,000 (the original, session-breaking scale) |

Mt. Hood's conservative plume-height range (dome-collapse style, capped at 24,000 m -- see intro) likely
doesn't disperse ash across the full 200 km domain regardless of resolution, so a smaller domain at moderate
spacing is probably both more physically appropriate *and* cheaper than simply coarsening the original extent.
Published Tephra2-based regional hazard assessments (e.g. Biass et al. 2016 / TephraProb) also typically use
grids in the hundreds-to-low-thousands range, not tens of thousands -- there's real precedent for that being
enough for a legitimate regional picture.

The cell prints the actual grid point count once it's built -- treat that as a checkpoint before committing to
a long sweep, not just the estimate above.

In [ ]:
vol_easting = round(vent_easting)
vol_northing = round(vent_northing)

grid_radius = 50_000    # meters either side of the vent -> domain = 2 * grid_radius square
grid_spacing = 5_000     # meters -- resolution
elevation = 1000         # meters, fixed (flat-topography assumption)

min_easting = vol_easting - grid_radius
max_easting = vol_easting + grid_radius
min_northing = vol_northing - grid_radius
max_northing = vol_northing + grid_radius

# Points of interest: Rhododendron, Parkdale, Govt. Camp
poi_names = ["Rhododendron", "Parkdale", "Govt. Camp"]
poi_locations = np.array([
    [45.329563, -121.911191],
    [45.519839, -121.596742],
    [45.30351444525525, -121.75331979925848],
])
poi_utm = utm.from_latlon(poi_locations[:, 0], poi_locations[:, 1])

n_grid_points = 0
with open("volcano_cone.grid", "w") as output_file:
    for i in range(min_easting, max_easting, grid_spacing):
        for j in range(min_northing, max_northing, grid_spacing):
            if i != vol_easting and j != vol_northing:
                print(i, j, elevation, file=output_file)
                n_grid_points += 1
    for i in range(poi_locations.shape[0]):
        print(f"{round(poi_utm[0][i])} {round(poi_utm[1][i])} {elevation}", file=output_file)

domain_km = 2 * grid_radius / 1000
print(f"Wrote volcano_cone.grid: {n_grid_points:,} regional grid points "
      f"({domain_km:.0f} x {domain_km:.0f} km @ {grid_spacing/1000:g} km spacing) + 3 POIs")

## Wind file

Wind velocity as a function of height, pulled from ERA5 reanalysis (`cdsapi`) for October 2024 and converted to
Tephra2's `height speed direction` format.

In [ ]:
# ERA5 request parameters
months = ["10"]
years = ["2024"]
days = ["23"]
hours = ["17:00"]

# ERA5 pressure levels (hPa) -- full set for vertical wind-profile resolution
pressures = ['1', '2', '3', '5', '7', '10', '20', '30', '50', '70',
             '100', '125', '150', '175', '200', '225', '250', '300',
             '350', '400', '450', '500', '550', '600', '650', '700',
             '750', '775', '800', '825', '850', '875', '900', '925',
             '950', '975', '1000']

north = round(vent_latitude * 4) / 4
south = north + 0.1
east = round(vent_longitude * 4) / 4
west = east - 0.1

wind_client = cdsapi.Client()
dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "data_format": "netcdf",
    "variable": ["geopotential", "u_component_of_wind", "v_component_of_wind"],
    "pressure_level": pressures,
    "year": years,
    "month": months,
    "day": days,
    "time": hours,
    "download_format": "unarchived",
    "area": [north, west, south, east],
}
wind_client.retrieve(dataset, request, "download.nc")


In [ ]:
wind = netCDF4.Dataset("download.nc")

uwnd = wind["u"][0, :, 0, 0]
vwnd = wind["v"][0, :, 0, 0]

speed = np.sqrt(uwnd**2 + vwnd**2)
direction = -180 / np.pi * np.arctan(vwnd / uwnd)
for d in range(len(direction)):
    if uwnd[d] > 0:
        direction[d] += 90
    else:
        direction[d] += 270

hgt = wind["z"][0, :, 0, 0] / 9.80665

# Tephra2 expects wind levels ordered from the ground up
speed = speed[::-1]
direction = direction[::-1]
hgt = hgt[::-1]

with open("my_wind.dat", "w") as wind_file:
    for level in range(wind["pressure_level"].shape[0]):
        wind_file.write(f"{hgt[level]} {speed[level]} {direction[level]}\n")

print("Wrote my_wind.dat")


## Parameter sweep ranges

Plume height, eruption mass, and diffusion coefficient are each swept over `n_steps` linearly spaced values,
giving `n_steps**3` Tephra2 runs. The plume height range is deliberately conservative given Mt. Hood's
dome-collapse eruptive style (see notebook intro) rather than spanning the full range typical of Plinian
volcanoes.

In [ ]:
n_steps = 10

ph_start, ph_end = 1_000, 24_000      # plume height (m asl)
em_start, em_end = 1e9, 1e12          # eruption mass (kg)
dc_start, dc_end = 1e3, 1e5           # diffusion coefficient (m^2/s)

plume_heights = np.linspace(ph_start, ph_end, n_steps, dtype=int)
eruption_masses = np.linspace(em_start, em_end, n_steps, dtype=int)
diffusion_coefs = np.linspace(dc_start, dc_end, n_steps, dtype=int)

total_runs = len(plume_heights) * len(eruption_masses) * len(diffusion_coefs)
print(f"{total_runs:,} total Tephra2 runs")


## Run the sweep

Each combination of `(plume_height, eruption_mass, diffusion_coef)` writes a fresh Tephra2 config file, runs
Tephra2 to a temporary CSV, reads that CSV, and appends the result to `tephra2_aggregated.csv`. Notes on how
this loop avoids known pitfalls:

- `row` (the config file's lines) is rebuilt from scratch **inside the innermost loop**, so one iteration's
  values can never leak into the next iteration's config file.
- `DIFFUSION_COEFFICIENT` is appended directly from the swept value — it is never conditionally overwritten by
  `eddy_const` or any other fallback.
- The per-run Tephra2 output (`tephra2_temp.csv`) is read immediately and deleted, and results are appended to
  the aggregated CSV incrementally (not held in memory), since a large sweep can produce far too many rows to
  hold in memory at once.
- Underflow values near zero (Tephra2 can emit values below double precision's usable range, e.g. `< 1e-300`)
  are written through as-is here; normalize them to exact zero in whatever reads `tephra2_aggregated.csv`.
- Latitude/longitude are **not** computed here, only `easting`/`northing` -- every run shares the same grid
  points (however many `grid_radius`/`grid_spacing` produce), so converting UTM -> lat/lon once per run would
  be pure repeated work. Do that conversion once downstream, over the unique `(easting, northing)` pairs, and
  join it back in (this is what `tephra2_analysis.ipynb` used to do for the old regional-grid design, before it
  was rewritten around the 3-POI CSVs -- see this notebook's intro for that gap).
- **Resumable**: on start, if `tephra2_aggregated.csv` already exists, it's scanned for which parameter
  combinations already have a *complete* set of grid rows (matched against `volcano_cone.grid`'s line count) --
  those are skipped. Any combination that's only partially written (interrupted mid-run) has its rows dropped
  and gets redone from scratch, so a resume never leaves duplicate or partial rows for one combination. This
  check re-reads the whole aggregated CSV, so it stays cheap only as long as `n_steps**3 * n_grid_points` stays
  small -- revisit with a chunked/DuckDB-based check before scaling either knob up a lot.

In [ ]:
agg_filename = "tephra2_aggregated.csv"
csv_columns = ['plume_height', 'eruption_mass', 'diffusion_coef', 'easting', 'northing', 'mass_kg_m2']

with open("volcano_cone.grid") as f:
    expected_points = sum(1 for _ in f)

completed_combos = set()
if os.path.exists(agg_filename):
    existing = pd.read_csv(agg_filename, on_bad_lines='skip')
    counts = existing.groupby(['plume_height', 'eruption_mass', 'diffusion_coef']).size()
    completed_combos = {tuple(int(x) for x in combo) for combo in counts[counts >= expected_points].index}

    # Drop rows from any combination that wasn't fully written (e.g. an interrupted kernel) so
    # redoing it below doesn't duplicate/mix in its leftover partial rows.
    keep = existing.set_index(['plume_height', 'eruption_mass', 'diffusion_coef']).index.isin(completed_combos)
    existing[keep].to_csv(agg_filename, index=False)
    print(f"Resuming: {len(completed_combos):,}/{total_runs:,} parameter combinations already completed")
else:
    with open(agg_filename, "w") as f:
        f.write(",".join(csv_columns) + "\n")

print_interval = max(1, total_runs // 20)
start_time = time.time()
run_count = len(completed_combos)
new_run_count = 0

for plume_height in plume_heights:
    for eruption_mass in eruption_masses:
        for diffusion_coef in diffusion_coefs:
            if (int(plume_height), int(eruption_mass), int(diffusion_coef)) in completed_combos:
                continue

            row = []

            row.append('VENT_EASTING ' + str(vent_easting))
            row.append('VENT_NORTHING ' + str(vent_northing))
            row.append('VENT_ELEVATION ' + str(vent_elevation))
            row.append('PLUME_HEIGHT ' + str(plume_height))
            row.append('ERUPTION_MASS ' + str(eruption_mass))
            row.append('MEDIAN_GRAINSIZE ' + str(median_grain))
            row.append('STD_GRAINSIZE ' + str(std_grain))
            row.append("".join(('FALL_TIME_THRESHOLD ', str(1000) if not falltime_thresh else str(falltime_thresh))))
            row.append("".join(('PLUME_MODEL ', str(2) if not plume_model else str(plume_model))))
            row.append("".join(('PART_STEPS ', str(100) if not part_steps else str(part_steps))))
            row.append("".join(('LITHIC_DENSITY ', str(2600.0) if not lithic_density else str(lithic_density))))
            row.append("".join(('COL_STEPS ', str(200) if not col_steps else str(col_steps))))
            row.append("".join(('MAX_GRAINSIZE ', str(-4) if not max_grain else str(max_grain))))
            row.append("".join(('ALPHA ', str(1) if not alpha else str(alpha))))
            row.append("".join(('BETA ', str(1) if not beta else str(beta))))
            row.append("".join(('MIN_GRAINSIZE ', str(4) if not min_grain else str(min_grain))))
            row.append("".join(('PUMICE_DENSITY ', str(1000.0) if not pumice_density else str(pumice_density))))
            row.append("".join(('EDDY_CONST ', str(0.04) if not eddy_const else str(eddy_const))))
            row.append('DIFFUSION_COEFFICIENT ' + str(diffusion_coef))

            with open("my_esps.conf", "w") as config_file:
                for element in row:
                    config_file.write(element + "\n")

            csv_filename = "tephra2_temp.csv"
            result = subprocess.run(
                f'{TEPHRA2_BIN} my_esps.conf volcano_cone.grid my_wind.dat > {csv_filename} 2>/dev/null',
                shell=True)
            os.remove("my_esps.conf")

            if result.returncode != 0:
                print(f"tephra2 failed for PH={plume_height} EM={eruption_mass} DC={diffusion_coef}")
                continue

            tephra_out = pd.read_csv(csv_filename, sep=r'\s+')
            tephra_out = tephra_out.rename(columns={'#EAST': 'easting', 'NORTH': 'northing', 'Kg/m^2': 'mass_kg_m2'})

            # Adding the three constant columns one at a time (tephra_out['x'] = ...) fragments the
            # DataFrame and triggers a PerformanceWarning on every single run; build them in one
            # pd.concat instead.
            params = pd.DataFrame({
                'plume_height': plume_height,
                'eruption_mass': eruption_mass,
                'diffusion_coef': diffusion_coef,
            }, index=tephra_out.index)
            out = pd.concat([params, tephra_out[['easting', 'northing', 'mass_kg_m2']]], axis=1)
            out.to_csv(agg_filename, mode='a', header=False, index=False)

            os.remove(csv_filename)

            run_count += 1
            new_run_count += 1
            if new_run_count % print_interval == 0:
                elapsed = time.time() - start_time
                rate = new_run_count / elapsed
                remaining = (total_runs - run_count) / rate if rate > 0 else float("inf")
                print(f"{run_count:,}/{total_runs:,} runs "
                      f"({100 * run_count / total_runs:.1f}%) — "
                      f"elapsed this session {elapsed/3600:.2f} h, est. remaining {remaining/3600:.2f} h")

print(f"Done. {run_count:,}/{total_runs:,} runs completed -> {agg_filename}")
